# 🧠 GNN Influencer Detection — Immoderma Skin Clinic Purwokerto
**Tugas Akhir | Prince Bayu Saputra | 2211103077 | Sistem Informasi — Universitas Telkom Purwokerto**

---
Pipeline lengkap: Data Instagram → Graf Jaringan Sosial → GCN / GAT / GIN → Identifikasi Influencer & Influencee

| Bagian | Isi |
|--------|-----|
| 1 | Setup & Instalasi |
| 2 | Load & Parsing Data |
| 3 | Konstruksi Graf Sosial |
| 4 | Analisis SNA (Baseline) |
| 5 | Feature Engineering & Labeling |
| 6 | Model GNN (GCN · GAT · GIN dengan skip-connection) |
| 7 | Evaluasi & Perbandingan |
| 8 | Identifikasi Influencer & Influencee |
| 9 | Visualisasi Jaringan |
| 10 | Export Hasil |


## ⚙️ 1. Instalasi & Setup

In [ ]:
# Jalankan cell ini pertama kali di Google Colab
import sys, subprocess

# Install torch-geometric (versi compatible Colab)
subprocess.run([sys.executable,'-m','pip','install',
    'torch-geometric','python-louvain','scikit-learn',
    'networkx','pandas','numpy','matplotlib','seaborn','-q'], check=False)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {DEVICE}")


## 📦 2. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import community as community_louvain
import re, warnings, json
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, GINConv
from torch_geometric.data import Data

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.titlesize':13})
print("✅ Semua library berhasil diimport!")


## 📥 3. Load Data Instagram

In [ ]:
from google.colab import files
import io

print("Upload file 'instagram.csv':")
uploaded = files.upload()
csv_name = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[csv_name]))
print(f"✅ '{csv_name}' berhasil diload — {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom")


In [ ]:
# ── Rename kolom scraping Instagram → nama deskriptif ─────────────────────────
col_map = {
    'x1i10hfl'   : 'akun_utama',
    '_ap3a'      : 'caption_post',
    'x1i10hfl 7' : 'username_commenter',
    '_ap3a 2'    : 'teks_komentar',
    'x193iq5w'   : 'reaksi_komentar',
    '_a9yi'      : 'balasan_komentar',
    'x1i10hfl 8' : 'username_reply_1',
}
df = df_raw.rename(columns=col_map)

# ── Fungsi helper ──────────────────────────────────────────────────────────────
def is_valid(u):
    """Filter username yang valid (bukan bot, bukan hashtag, bukan akun klinik)."""
    if pd.isna(u): return False
    u = str(u).strip().lower()
    if u in ['immoderma_purwokerto', 'suka', '']: return False
    if re.fullmatch(r'\d+', u): return False
    if u.startswith('#'): return False
    return True

def parse_likes(v):
    m = re.match(r'^(\d+)\s*suka', str(v).strip().lower()) if not pd.isna(v) else None
    return int(m.group(1)) if m else 0

def parse_replies(v):
    m = re.search(r'\((\d+)\)', str(v)) if not pd.isna(v) else None
    return int(m.group(1)) if m else 0

# ── Ekstrak baris komentar yang bersih ────────────────────────────────────────
df_c = df[df['username_commenter'].notna() & df['teks_komentar'].notna()].copy()
df_c = df_c[df_c['username_commenter'].apply(is_valid)].copy()
df_c = df_c.drop_duplicates(subset=['username_commenter', 'teks_komentar']).reset_index(drop=True)
df_c['comment_likes'] = df_c['reaksi_komentar'].apply(parse_likes)
df_c['reply_count']   = df_c['balasan_komentar'].apply(parse_replies)

print(f"✅ Komentar bersih   : {len(df_c):,} baris")
print(f"   Unique commenter  : {df_c['username_commenter'].nunique():,}")
print(f"\nSample data:")
display(df_c[['username_commenter','teks_komentar','comment_likes','reply_count']].head(8))


## 🕸️ 4. Konstruksi Graf Jaringan Sosial

In [ ]:
# ── Logika bobot edge ─────────────────────────────────────────────────────────
# Komentar user → klinik      : bobot 1
# Balasan klinik → user       : bobot 2  (interaksi lebih kuat)
# Interaksi user ↔ user       : bobot 3  (hubungan komunitas terkuat)

KLINIK = 'immoderma_purwokerto'
G = nx.DiGraph()
G.add_node(KLINIK, node_type='klinik')

for _, r in df_c.iterrows():
    u = r['username_commenter']
    if not G.has_node(u): G.add_node(u, node_type='customer')

    # User → Klinik
    if G.has_edge(u, KLINIK): G[u][KLINIK]['weight'] += 1
    else: G.add_edge(u, KLINIK, weight=1, edge_type='comment')

    # Klinik → User (jika ada balasan)
    if r['reply_count'] > 0:
        if G.has_edge(KLINIK, u): G[KLINIK][u]['weight'] += 2
        else: G.add_edge(KLINIK, u, weight=2, edge_type='clinic_reply')

    # User → User (via kolom reply chain)
    r1 = r['username_reply_1']
    if is_valid(r1) and r1 != u:
        if not G.has_node(r1): G.add_node(r1, node_type='customer')
        if G.has_edge(u, r1): G[u][r1]['weight'] += 3
        else: G.add_edge(u, r1, weight=3, edge_type='user_interaction')

customers = [n for n in G.nodes() if n != KLINIK]

print("=" * 50)
print("  STATISTIK GRAF JARINGAN SOSIAL")
print("=" * 50)
print(f"  Total node       : {G.number_of_nodes():,}")
print(f"  Total edge       : {G.number_of_edges():,}")
print(f"  Customer nodes   : {len(customers):,}")
print(f"  Directed         : {G.is_directed()}")
print(f"  Density          : {nx.density(G):.5f}")

# Komponen
comps = list(nx.weakly_connected_components(G))
print(f"  Komponen         : {len(comps)} (terbesar: {max(len(c) for c in comps):,} node)")


## 📐 5. Analisis SNA Konvensional (Baseline)

In [ ]:
print("⏳ Menghitung metrik sentralitas...")

in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())
in_wdeg  = {n: sum(d.get('weight',1) for _,_,d in G.in_edges(n,data=True)) for n in G.nodes()}
out_wdeg = {n: sum(d.get('weight',1) for _,_,d in G.out_edges(n,data=True)) for n in G.nodes()}
deg_c  = nx.in_degree_centrality(G)
btwn   = nx.betweenness_centrality(G, normalized=True, k=min(400, G.number_of_nodes()))
pr     = nx.pagerank(G, alpha=0.85)
lcc    = max(nx.weakly_connected_components(G), key=len)
clos   = nx.closeness_centrality(G.subgraph(lcc))

sna = pd.DataFrame({'username': customers})
for col, d in [('in_degree',in_deg),('out_degree',out_deg),
               ('in_wdeg',in_wdeg),('out_wdeg',out_wdeg),
               ('degree_c',deg_c),('betweenness',btwn),
               ('pagerank',pr),('closeness',clos)]:
    sna[col] = sna['username'].map(d).fillna(0)

us = df_c.groupby('username_commenter').agg(
    n_comments   = ('teks_komentar','count'),
    total_likes  = ('comment_likes','sum'),
    total_replies= ('reply_count','sum'),
).reset_index().rename(columns={'username_commenter':'username'})
sna = sna.merge(us, on='username', how='left').fillna(0)
sna['engagement_score']   = sna['n_comments']*1 + sna['total_likes']*2 + sna['total_replies']*3
sna['reply_per_comment']  = np.where(sna['n_comments']>0, sna['total_replies']/sna['n_comments'], 0)

print("✅ Metrik sentralitas selesai!")
print("\nTop 15 node berdasarkan PageRank + In-Degree:")
display(sna.sort_values('pagerank', ascending=False)
          [['username','in_degree','betweenness','pagerank','n_comments','total_replies','engagement_score']]
          .head(15).round(6))


In [ ]:
# ── Deteksi Komunitas Louvain ─────────────────────────────────────────────────
G_und = G.to_undirected()
partition  = community_louvain.best_partition(G_und, random_state=SEED)
modularity = community_louvain.modularity(partition, G_und)
comm_sizes = Counter(partition.values())

print(f"✅ Komunitas Louvain: {len(comm_sizes)} komunitas | Modularity = {modularity:.4f}")

# Visualisasi distribusi ukuran komunitas (top 20)
top_comms = sorted(comm_sizes.values(), reverse=True)[:20]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(range(len(top_comms)), top_comms, color='#3498db', edgecolor='white')
axes[0].set_title(f'Ukuran 20 Komunitas Terbesar\n(Total {len(comm_sizes)} komunitas, Modularity={modularity:.4f})', fontweight='bold')
axes[0].set_xlabel('Rank Komunitas'); axes[0].set_ylabel('Jumlah Node')

# Distribusi sentralitas
sna_nonzero = sna[sna['pagerank'] > sna['pagerank'].median()]
axes[1].scatter(sna_nonzero['betweenness'], sna_nonzero['pagerank'],
                c=sna_nonzero['engagement_score'], cmap='YlOrRd', alpha=0.7, s=30)
axes[1].set_xlabel('Betweenness Centrality'); axes[1].set_ylabel('PageRank')
axes[1].set_title('Peta Sentralitas Node\n(warna = engagement score)', fontweight='bold')
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Engagement Score')

plt.tight_layout()
plt.savefig('sna_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 🧮 6. Feature Engineering & Pelabelan

In [ ]:
# ── Buat label ground truth berbasis SNA Composite (top 5%) ──────────────────
# Composite score: rata-rata rank percentile dari degree, betweenness, pagerank + engagement
sna['sna_composite'] = (
    sna['degree_c'].rank(pct=True)    * 0.30 +
    sna['betweenness'].rank(pct=True) * 0.30 +
    sna['pagerank'].rank(pct=True)    * 0.30 +
    sna['engagement_score'].rank(pct=True) * 0.10
)

threshold = sna['sna_composite'].quantile(0.95)
sna['label'] = (sna['sna_composite'] >= threshold).astype(int)

n_inf = sna['label'].sum()
n_non = (sna['label']==0).sum()
print(f"Label Influencer (1) : {n_inf} ({100*n_inf/len(sna):.1f}%)")
print(f"Label Non-Inf    (0) : {n_non} ({100*n_non/len(sna):.1f}%)")
print(f"Imbalance ratio      : 1:{n_non/n_inf:.1f}")

# ── Fitur node (11 dimensi) ────────────────────────────────────────────────────
FEAT_COLS = ['in_degree','out_degree','in_wdeg','out_wdeg',
             'degree_c','betweenness','pagerank','closeness',
             'n_comments','engagement_score','reply_per_comment']

X = StandardScaler().fit_transform(sna[FEAT_COLS].values.astype(float))
y = sna['label'].values
N = len(y)

print(f"\nFeature matrix : {X.shape}")
print(f"Fitur          : {FEAT_COLS}")

# Visualisasi distribusi label
fig, ax = plt.subplots(figsize=(7,4))
bars = ax.bar(['Non-Influencer','Influencer'], [n_non, n_inf],
              color=['#95a5a6','#e74c3c'], edgecolor='white', width=0.4)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+5, str(int(b.get_height())),
            ha='center', fontweight='bold', fontsize=12)
ax.set_title('Distribusi Label Node (Ground Truth)', fontweight='bold')
ax.set_ylabel('Jumlah Node')
plt.tight_layout(); plt.savefig('label_dist.png', dpi=150, bbox_inches='tight'); plt.show()


## 🔧 7. Konversi ke PyTorch Geometric

In [ ]:
# ── Edge index ────────────────────────────────────────────────────────────────
node_list = sna['username'].tolist()
n2i = {n:i for i,n in enumerate(node_list)}

src_l, tgt_l = [], []
for u, v in G.edges():
    if u in n2i and v in n2i:
        src_l.append(n2i[u]); tgt_l.append(n2i[v])

edge_index = torch.tensor([src_l, tgt_l], dtype=torch.long)
x_t = torch.tensor(X, dtype=torch.float)
y_t = torch.tensor(y, dtype=torch.long)

# ── Train / Val / Test split (70 : 15 : 15, stratified) ───────────────────────
idx = np.arange(N)
tr_idx, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)
vl_idx, te_idx = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=SEED)

tr_m = torch.zeros(N, dtype=torch.bool); tr_m[tr_idx] = True
vl_m = torch.zeros(N, dtype=torch.bool); vl_m[vl_idx] = True
te_m = torch.zeros(N, dtype=torch.bool); te_m[te_idx] = True

# Class weight untuk imbalance
cw = compute_class_weight('balanced', classes=np.unique(y[tr_idx]), y=y[tr_idx])
cw_t = torch.tensor(cw, dtype=torch.float)

print(f"Edge index shape : {edge_index.shape}")
print(f"Node features    : {x_t.shape}")
print(f"Train : {tr_m.sum().item()}, Val : {vl_m.sum().item()}, Test : {te_m.sum().item()}")
print(f"Class weights    : Non-Inf={cw[0]:.3f}, Influencer={cw[1]:.3f}")


## 🤖 8. Arsitektur Model GNN

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Semua arsitektur menggunakan:
#   • 3 conv layer  + BatchNorm  + Dropout(0.4)
#   • Skip-connection (residual) dari input ke output conv terakhir
#   • MLP head: Linear → ReLU → Dropout → Linear
# ────────────────────────────────────────────────────────────────────────────

class ImprovedGCN(nn.Module):
    """Graph Convolutional Network dengan skip-connection."""
    def __init__(self, in_c, hid, out_c, drop=0.4):
        super().__init__()
        self.conv1 = GCNConv(in_c, hid); self.conv2 = GCNConv(hid, hid); self.conv3 = GCNConv(hid, hid)
        self.bn1 = nn.BatchNorm1d(hid); self.bn2 = nn.BatchNorm1d(hid); self.bn3 = nn.BatchNorm1d(hid)
        self.skip = nn.Linear(in_c, hid)
        self.head = nn.Sequential(nn.Linear(hid, hid//2), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid//2, out_c))
        self.drop = drop
    def forward(self, x, ei):
        skip = self.skip(x)
        h = F.dropout(F.relu(self.bn1(self.conv1(x, ei))), p=self.drop, training=self.training)
        h = F.dropout(F.relu(self.bn2(self.conv2(h, ei))), p=self.drop, training=self.training)
        h = F.relu(self.bn3(self.conv3(h, ei))) + skip
        return self.head(h)


class ImprovedGAT(nn.Module):
    """Graph Attention Network (4 heads) dengan skip-connection."""
    def __init__(self, in_c, hid, out_c, heads=4, drop=0.4):
        super().__init__()
        self.conv1 = GATConv(in_c, hid, heads=heads, dropout=drop, concat=True)
        self.conv2 = GATConv(hid*heads, hid, heads=heads, dropout=drop, concat=True)
        self.conv3 = GATConv(hid*heads, hid, heads=1, dropout=drop, concat=False)
        self.bn1 = nn.BatchNorm1d(hid*heads); self.bn2 = nn.BatchNorm1d(hid*heads); self.bn3 = nn.BatchNorm1d(hid)
        self.skip = nn.Linear(in_c, hid)
        self.head = nn.Sequential(nn.Linear(hid, hid//2), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid//2, out_c))
        self.drop = drop
    def forward(self, x, ei):
        skip = self.skip(x)
        h = F.dropout(F.elu(self.bn1(self.conv1(x, ei))), p=self.drop, training=self.training)
        h = F.dropout(F.elu(self.bn2(self.conv2(h, ei))), p=self.drop, training=self.training)
        h = F.elu(self.bn3(self.conv3(h, ei))) + skip
        return self.head(h)


class ImprovedGIN(nn.Module):
    """Graph Isomorphism Network dengan skip-connection dan MLP internal."""
    def __init__(self, in_c, hid, out_c, drop=0.4):
        super().__init__()
        def mlp(a, b):
            return nn.Sequential(nn.Linear(a,b), nn.BatchNorm1d(b), nn.ReLU(),
                                 nn.Dropout(drop), nn.Linear(b,b), nn.BatchNorm1d(b))
        self.conv1 = GINConv(mlp(in_c, hid), train_eps=True)
        self.conv2 = GINConv(mlp(hid, hid), train_eps=True)
        self.conv3 = GINConv(mlp(hid, hid), train_eps=True)
        self.skip = nn.Linear(in_c, hid)
        self.head = nn.Sequential(nn.Linear(hid, hid//2), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid//2, out_c))
        self.drop = drop
    def forward(self, x, ei):
        skip = self.skip(x)
        h = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        h = F.dropout(F.relu(self.conv2(h, ei)), p=self.drop, training=self.training)
        h = F.relu(self.conv3(h, ei)) + skip
        return self.head(h)


IN = len(FEAT_COLS); HID = 128; OUT = 2
print(f"✅ Arsitektur: IN={IN} → HIDDEN={HID} → OUT={OUT}")
print("   GCN : GCNConv × 3 + skip + MLP head")
print("   GAT : GATConv(heads=4) × 3 + skip + MLP head")
print("   GIN : GINConv(MLP) × 3 + skip + MLP head")


## 🏋️ 9. Training & Evaluasi

In [ ]:
def train_model(ModelCls, kwargs, name, epochs=300, lr=5e-4, wd=1e-4, patience=50):
    """Training loop dengan early stopping + LR scheduler."""
    model = ModelCls(**kwargs)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=20, factor=0.5)
    criterion = nn.CrossEntropyLoss(weight=cw_t)

    best_vf1, best_state, no_improve = 0, None, 0
    loss_hist, vf1_hist = [], []

    for ep in range(1, epochs + 1):
        # ── Train ────────────────────────────────────────────────────────────
        model.train(); optimizer.zero_grad()
        out  = model(x_t, edge_index)
        loss = criterion(out[tr_m], y_t[tr_m])
        loss.backward(); optimizer.step()

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            out2 = model(x_t, edge_index)
            vf1  = f1_score(y_t[vl_m].numpy(), out2[vl_m].argmax(1).numpy(), zero_division=0)

        scheduler.step(vf1)
        loss_hist.append(round(loss.item(), 4))
        vf1_hist.append(round(vf1, 4))

        if vf1 > best_vf1:
            best_vf1 = vf1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"  [{name}] Early stop epoch={ep} | Best Val F1={best_vf1:.4f}")
            break

    # ── Test evaluation ───────────────────────────────────────────────────────
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        out = model(x_t, edge_index)
        yp   = out[te_m].argmax(1).numpy()
        yprob = F.softmax(out[te_m], dim=1)[:,1].numpy()

    yt = y_t[te_m].numpy()
    try: auc_v = round(roc_auc_score(yt, yprob), 4)
    except: auc_v = 0.0

    result = {
        'name'      : name,
        'accuracy'  : round(accuracy_score(yt, yp), 4),
        'precision' : round(precision_score(yt, yp, zero_division=0), 4),
        'recall'    : round(recall_score(yt, yp, zero_division=0), 4),
        'f1'        : round(f1_score(yt, yp, zero_division=0), 4),
        'auc'       : auc_v,
        'yt':yt, 'yp':yp, 'yprob':yprob,
        'loss_hist' : loss_hist,
        'vf1_hist'  : vf1_hist,
    }
    print(f"[{name}] Acc={result['accuracy']} | Prec={result['precision']} | Rec={result['recall']} | F1={result['f1']} | AUC={result['auc']}")
    return model, result

print("✅ Fungsi training siap. Mulai pelatihan ketiga model..\n")

# ── Jalankan training ─────────────────────────────────────────────────────────
print("=" * 50 + "\n  GCN\n" + "=" * 50)
gcn_m, gcn_r = train_model(ImprovedGCN, dict(in_c=IN, hid=HID, out_c=OUT), 'GCN')

print("\n" + "=" * 50 + "\n  GAT\n" + "=" * 50)
gat_m, gat_r = train_model(ImprovedGAT, dict(in_c=IN, hid=HID, out_c=OUT, heads=4), 'GAT')

print("\n" + "=" * 50 + "\n  GIN\n" + "=" * 50)
gin_m, gin_r = train_model(ImprovedGIN, dict(in_c=IN, hid=HID, out_c=OUT), 'GIN')

print("\n✅ Pelatihan selesai!")


## 📊 10. Perbandingan Model & Visualisasi

In [ ]:
# ── Baseline SNA ─────────────────────────────────────────────────────────────
te_sna = sna.iloc[te_idx]
yt_te  = y[te_idx]

def baseline_metrics(col, yt, sna_df):
    thresh = np.percentile(sna_df[col].values, 100*(1 - yt.mean()))
    yp = (sna_df[col].values >= thresh).astype(int)
    try: auc_v = round(roc_auc_score(yt, sna_df[col].values), 4)
    except: auc_v = 0.0
    return {'accuracy': round(accuracy_score(yt,yp),4),
            'precision': round(precision_score(yt,yp,zero_division=0),4),
            'recall': round(recall_score(yt,yp,zero_division=0),4),
            'f1': round(f1_score(yt,yp,zero_division=0),4), 'auc': auc_v}

bdc = baseline_metrics('degree_c', yt_te, te_sna)
bbt = baseline_metrics('betweenness', yt_te, te_sna)
bpr = baseline_metrics('pagerank', yt_te, te_sna)

# ── Tabel Perbandingan ────────────────────────────────────────────────────────
compare = pd.DataFrame([
    {'Model': 'Degree Centrality (Baseline)',     **bdc},
    {'Model': 'Betweenness Centrality (Baseline)',**bbt},
    {'Model': 'PageRank (Baseline)',              **bpr},
    {'Model': 'GCN (Proposed)',                  **{k:gcn_r[k] for k in ['accuracy','precision','recall','f1','auc']}},
    {'Model': 'GAT (Proposed)',                  **{k:gat_r[k] for k in ['accuracy','precision','recall','f1','auc']}},
    {'Model': 'GIN (Proposed) ★',               **{k:gin_r[k] for k in ['accuracy','precision','recall','f1','auc']}},
])

print("\n📊 TABEL PERBANDINGAN PERFORMA MODEL")
print("=" * 72)
display(compare.style
    .highlight_max(subset=['accuracy','precision','recall','f1','auc'], color='#d5f5e3')
    .format({'accuracy':'{:.4f}','precision':'{:.4f}','recall':'{:.4f}','f1':'{:.4f}','auc':'{:.4f}'})
)


In [ ]:
# ── Learning Curves ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
model_res = [('GCN','#3498db',gcn_r), ('GAT','#e67e22',gat_r), ('GIN','#e74c3c',gin_r)]

for name, color, res in model_res:
    axes[0].plot(res['loss_hist'], label=name, color=color, lw=2, alpha=0.85)
    axes[1].plot(res['vf1_hist'],  label=name, color=color, lw=2, alpha=0.85)

axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Validation F1-Score', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1-Score'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(0.8, ls='--', color='gray', alpha=0.5, label='F1=0.8')

plt.suptitle('Learning Curves — GCN vs GAT vs GIN', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight'); plt.show()


In [ ]:
# ── Confusion Matrix + ROC Curve (Model Terbaik = GIN) ───────────────────────
best_r = gin_r  # GIN terbaik

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(best_r['yt'], best_r['yp'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Non-Influencer','Influencer'],
            yticklabels=['Non-Influencer','Influencer'], linewidths=0.5)
axes[0].set_title('Confusion Matrix — GIN (Model Terbaik)', fontweight='bold')
axes[0].set_xlabel('Prediksi'); axes[0].set_ylabel('Aktual')

# ROC Curve semua model
colors = {'GCN':'#3498db','GAT':'#e67e22','GIN':'#e74c3c'}
for name, color, res in model_res:
    fpr, tpr, _ = roc_curve(res['yt'], res['yprob'])
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={res['auc']:.4f})")
axes[1].plot([0,1],[0,1],'--',color='gray',label='Random')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('confusion_roc.png', dpi=150, bbox_inches='tight'); plt.show()
print("\nClassification Report — GIN:")
print(classification_report(best_r['yt'], best_r['yp'], target_names=['Non-Influencer','Influencer'], zero_division=0))


## 🎯 11. Identifikasi Influencer & Influencee

In [ ]:
# ── Inferensi penuh semua node ───────────────────────────────────────────────
best_model = gin_m  # GIN
best_model.eval()
with torch.no_grad():
    out_all   = best_model(x_t, edge_index)
    probs_all = F.softmax(out_all, dim=1)[:, 1].numpy()

sna['prob_inf'] = probs_all

# ── Kategorisasi Tier ─────────────────────────────────────────────────────────
tier1 = sna[sna['prob_inf'] >= 0.70].sort_values('prob_inf', ascending=False)
tier2 = sna[(sna['prob_inf'] >= 0.50) & (sna['prob_inf'] < 0.70)].sort_values('prob_inf', ascending=False)
non   = sna[sna['prob_inf'] < 0.50]

print("📊 Distribusi Tier:")
print(f"   Influencer Tier 1 (prob ≥ 0.70) : {len(tier1):,}")
print(f"   Influencer Tier 2 (prob ≥ 0.50) : {len(tier2):,}")
print(f"   Non-Influencer    (prob < 0.50)  : {len(non):,}")

print("\n🔴 TOP 20 INFLUENCER TIER 1:")
display(tier1[['username','prob_inf','in_degree','betweenness','pagerank',
               'n_comments','total_replies','engagement_score']].head(20).round(5))


In [ ]:
# ── Identifikasi Influencee (Target Promosi) ─────────────────────────────────
# Influencee = node non-influencer yang terhubung langsung (1-hop) ke influencer

inf_set = set(tier1['username']) | set(tier2['username'])
influencee_list = []
for node in G.nodes():
    if node == KLINIK or node in inf_set or node not in n2i:
        continue
    neighbors = (set(G.predecessors(node)) | set(G.successors(node))) & inf_set
    if neighbors:
        influencee_list.append({
            'username'              : node,
            'n_influencer_terhubung': len(neighbors),
            'influencer_terhubung'  : ', '.join(list(neighbors)[:3]),
            'prob_influencer'       : round(float(probs_all[n2i[node]]), 4),
        })

inf_df = pd.DataFrame(influencee_list).sort_values(
    ['n_influencer_terhubung','prob_influencer'], ascending=False
).reset_index(drop=True)

print(f"🟡 Total Influencee (Target Promosi) : {len(inf_df):,}")
print("\nTop 20 Influencee:")
display(inf_df.head(20))


## 📈 12. Visualisasi Jaringan Sosial

In [ ]:
# ── Ambil subgraph paling relevan ────────────────────────────────────────────
key_nodes = (
    {KLINIK} |
    set(tier1['username'].head(15)) |
    set(tier2['username'].head(10)) |
    set(inf_df['username'].head(15))
)
# Tambah 1-hop neighbors
extra = set()
for n in key_nodes:
    if n in G:
        extra.update(list(G.predecessors(n))[:3])
        extra.update(list(G.successors(n))[:3])
lcc_nodes = max(nx.weakly_connected_components(G), key=len)
G_plot = G.subgraph((key_nodes | extra) & lcc_nodes).copy()

def node_color(n):
    if n == KLINIK: return '#2c3e50'
    if n in set(tier1['username']): return '#e74c3c'
    if n in set(tier2['username']): return '#e67e22'
    if n in set(inf_df['username']): return '#f39c12'
    return '#bdc3c7'

def node_size(n):
    if n == KLINIK: return 1500
    if n in set(tier1['username']): return 600
    if n in set(tier2['username']): return 350
    if n in set(inf_df['username']): return 180
    return 80

nc = [node_color(n) for n in G_plot.nodes()]
ns = [node_size(n) for n in G_plot.nodes()]
pos = nx.spring_layout(G_plot, seed=SEED, k=2.0)

fig, ax = plt.subplots(figsize=(18, 14))
fig.patch.set_facecolor('#0f0f1a'); ax.set_facecolor('#0f0f1a')

nx.draw_networkx_edges(G_plot, pos, ax=ax, alpha=0.2, edge_color='#7f8c8d',
                       arrows=True, arrowsize=6, connectionstyle='arc3,rad=0.08')
nx.draw_networkx_nodes(G_plot, pos, ax=ax, node_color=nc, node_size=ns, alpha=0.9)
labels = {n: n for n in G_plot.nodes() if n in key_nodes}
nx.draw_networkx_labels(G_plot, pos, labels=labels, ax=ax, font_size=6, font_color='white')

legend_patches = [
    mpatches.Patch(color='#2c3e50', label=f'Klinik Immoderma'),
    mpatches.Patch(color='#e74c3c', label=f'Influencer Tier 1 (n={len(tier1)})'),
    mpatches.Patch(color='#e67e22', label=f'Influencer Tier 2 (n={len(tier2)})'),
    mpatches.Patch(color='#f39c12', label=f'Influencee / Target Promosi (n={len(inf_df)})'),
    mpatches.Patch(color='#bdc3c7', label='Pelanggan Reguler'),
]
ax.legend(handles=legend_patches, loc='upper left', facecolor='#1a1a2e', labelcolor='white', fontsize=10, framealpha=0.9)
ax.set_title('Peta Jaringan Sosial Pelanggan — Immoderma Skin Clinic Purwokerto',
             color='white', fontsize=15, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()
plt.savefig('network_map.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print(f"✅ Peta jaringan: {G_plot.number_of_nodes()} node, {G_plot.number_of_edges()} edge")


## 💾 13. Export Hasil

In [ ]:
import zipfile, os
from google.colab import files

# CSV
tier1.to_csv('influencer_tier1.csv', index=False)
tier2.to_csv('influencer_tier2.csv', index=False)
inf_df.to_csv('influencee_target_promosi.csv', index=False)
compare.to_csv('perbandingan_model.csv', index=False)
sna.to_csv('node_features_lengkap.csv', index=False)

# ZIP
zip_name = 'hasil_GNN_Immoderma_Prince_2211103077.zip'
file_list = ['influencer_tier1.csv','influencer_tier2.csv','influencee_target_promosi.csv',
             'perbandingan_model.csv','node_features_lengkap.csv','network_map.png',
             'learning_curves.png','confusion_roc.png','sna_analysis.png','label_dist.png']
with zipfile.ZipFile(zip_name, 'w') as zf:
    for f in file_list:
        if os.path.exists(f): zf.write(f)

print("✅ Semua file dikemas! Isi ZIP:")
with zipfile.ZipFile(zip_name,'r') as zf:
    for name in zf.namelist(): print(f"   {name}")
files.download(zip_name)


## 📋 14. Ringkasan Hasil

In [ ]:
print("=" * 65)
print("  RINGKASAN HASIL — GNN Influencer Detection")
print("  Immoderma Skin Clinic Purwokerto")
print(f"  Peneliti : Prince Bayu Saputra / 2211103077")
print("=" * 65)

print(f"\n📊 DATASET:")
print(f"   Komentar bersih     : {len(df_c):,}")
print(f"   Node (customer)     : {len(customers):,}")
print(f"   Edge (interaksi)    : {G.number_of_edges():,}")
print(f"   Density             : {nx.density(G):.5f}")

print(f"\n🔍 KOMUNITAS (Louvain):")
print(f"   Jumlah komunitas    : {len(comm_sizes)}")
print(f"   Modularity          : {modularity:.4f}")

best = gin_r
print(f"\n🏆 MODEL TERBAIK: GIN (Graph Isomorphism Network)")
print(f"   Accuracy  : {best['accuracy']:.4f}")
print(f"   Precision : {best['precision']:.4f}")
print(f"   Recall    : {best['recall']:.4f}")
print(f"   F1-Score  : {best['f1']:.4f}")
print(f"   AUC-ROC   : {best['auc']:.4f}")

print(f"\n🎯 IDENTIFIKASI:")
print(f"   Influencer Tier 1   : {len(tier1):,} akun")
print(f"   Influencer Tier 2   : {len(tier2):,} akun")
print(f"   Influencee (target) : {len(inf_df):,} akun")
print("=" * 65)
